# Platonic Earth: run the full website in Colab

This notebook runs the **backend and the website** in Colab, so you can pick **any place on Earth**, set a **radius**, and search the live Sentinel-2 imagery inside it in plain English.

**Nothing is trained here.** It loads the rotation we already fitted on an RTX 5060 Ti: OSM-Sentinel, **1,000,000** image–caption pairs from the 1M build (`colab/artifacts/osm_1m_rotation.npz`, SHA-256 `358c5b94aa58…`). It uses the same frozen DINOv2-small and all-MiniLM-L6-v2 models.

**How to use it:** run the cells in order (Runtime → Run all). A GPU makes the image embedding faster but is optional. When the last cell finishes, open the link it prints. In the site:
1. Scroll to **Chapter 03 · Any city** (the Lab).
2. **Search a place**, press **Radius**, set the distance (up to 6 km), and click the map to move the circle.
3. Press **Analyse area**. It downloads the clearest recent Sentinel-2 pass and labels every 640 m square with our rotation (about 15–60 s).
4. Use the **Ask** tab, e.g. "parking lots and big roofs". Only the squares inside your circle are ranked.

*Prototype, not a land-use determination.*

## 1 · Get the code

In [ ]:
import os, subprocess
if not os.path.exists("/content/veritas"):
    !git clone --depth 1 --branch a/web-area-search https://github.com/JohnPoole3rd/veritas.git /content/veritas
%cd /content/veritas
!git log --oneline -1

## 2 · Install the dependencies (about 1–2 minutes)

In [ ]:
!pip install -q -r requirements.txt
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine)")

## 3 · Put the already-fitted artifacts in place
This checks the rotation's checksum and copies three small precomputed embedding files to where the backend reads them. No training.

In [ ]:
!python colab/prepare.py

## 4 · Download the two frozen models once (DINOv2-small, all-MiniLM-L6-v2)

In [ ]:
!python app.py --prefetch

## 5 · Start the backend with the 1M OSM rotation and open the website

In [ ]:
import time, urllib.request, json
env = dict(os.environ, PE_ROTATION="colab/artifacts/osm_1m_rotation.npz")
log = open("/content/server.log", "w")
server = subprocess.Popen(["python", "-m", "web.server", "--port", "8000", "--host", "127.0.0.1", "--no-classic", "--online"],
                          cwd="/content/veritas", env=env, stdout=log, stderr=subprocess.STDOUT)
meta = None
for _ in range(120):
    try:
        meta = json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/meta", timeout=5)); break
    except Exception:
        if server.poll() is not None: break
        time.sleep(3)
if meta is None:
    print(open("/content/server.log").read()[-3000:]); raise SystemExit("The backend did not start; see the log above.")
print("Backend:", meta["mode"], "| device:", meta["device"])
print("Rotation:", meta["rotation"])
assert meta["mode"] == "REAL" and meta["rotation"]["sha256"] == "358c5b94aa58", "not running the 1M rotation"

from google.colab import output
output.serve_kernel_port_as_window(8000, path="/")

Click the link above to open the website in a new tab. To show it inside the notebook instead, run the next cell.

**If something goes wrong:** `!tail -50 /content/server.log` shows the backend log. The site needs internet access for the satellite imagery (Element 84 Earth Search, Microsoft Planetary Computer for Landsat) and the basemap. A study area is capped at 400 squares of 640 m, so a radius of 6 km or less fits.

**Note:** the "Results" chapter of the site shows the numbers from our logged EuroSAT runs. The Lab, which is what this notebook is for, uses the 1M OSM rotation, and the server confirms it at `/api/meta` → `rotation`.

In [ ]:
output.serve_kernel_port_as_iframe(8000, path="/", height=900)